<a href="https://colab.research.google.com/github/doralalam/llm-engineering/blob/notes/003_week/day_4/030_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### **Models**

- We are now looking at the lower level API of transformers - the models that wrap the PyTorch code for the transformers themselves

#### **Quantization**

- Quantization is the process of shrinking the memory footprint of a model by compressing the parameters from 16-bit to 4-bit or 8-bit without significant drop in performance

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


#### Breakdown

- `bitsandbytes` - a lightweight CUDA wrapper library that enables 4-bit and 8-bit model quantization

  - Standard 16-bit LLMSs requires massive amounts of GPU memory (vRAM)

  - `bitsandbytes` shrinks the model weight precision down to 8-bit or 4-bit (NF4 format) without significant loss in accuracy.

  - This enables loading large models onto smaller GPUs

- `accelerate` - HuggingFace library designed to simply run the models across single-GPU, multi-GPU  or CPU setup configurations

  - If an LLM is too large to run on a single GPU's vRAM, accelerate handles the splitting the model's layers across multiple GPUs or spilling excess layers onto the system CPU RAM

In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

#### Breakdown

- `AutoTokenizer` - Automatically selects and loads the correct tokenizer for any model architecture

- `AutoModelForCausalLM` - Automatically loads the model architecture configured for causal language modelling

- `TextStreamer` - Enables real-time text streaming (token-by-token) rather than waiting for the entire response to complete before printing

- `BitsAndBytesConfig` - Allows us to config 4-bit or 8-bit quantization via bitsandbytes library

- `torch` - Provides the underlying tensor operations

- `gc` - Garbage Collector to manually force the release of unused memory

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [4]:
# instruct models and 1 reasoning model

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [5]:
messages = [
    {'role':'user', 'content':'Tell me a joke for the upcoming Data Scientists'}
    ]

In [6]:
# Quantization Config - This allows us to load the large model on to the memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4_bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type = 'nf4'
)

#### Breakdown

- The above code snipped configures QLoRA-style-4bit-quantization using HuggingFace's BitsAndBytesConfig utility from transformers library

- It shrinks an LLM's memory footprint by ~75% (ex: bringing a 14GB model down to ~3.5GB model) so it can fit on consumers smaller GPU with minimal loss

- `load_in_4bit = True` - Enables 4-bit quantization. When loading model weights from HuggingFace, linear layer parameters are compressed from 16-bit floats to 4-bit integers

- `bnb_4bit_use_double_quant = True` - Applies a second layer of quantization to store strong `quantization constants` with zero loss in performace

- `bnb_4bit_compute_dtype = torch.bfloat16` - Model weights are stored on the GPU in 4-bit, but actual mathematical matrix operations (activations and gradients) cannot run natively in 4-bit.

  - This tells PyTorch to dynamically dequantize the parameters into `bfloat16` during computation

  - `bfloat16` prevents underflow / overflow issues during forward and backward passes, providing higher numerical stability than standard `float16` on newer GPUs

- `bnb_4bit_quant_type = 'nf4'`

  - standard quantization uses `fp4` which causes higher precision loss because neural networks follow Gaussing Distribution

  - `nf4` - maps to normal distribution yielding better quality than standard fp4


In [7]:
# tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
input = tokenizer.apply_chat_template(messages, return_tensors='pt').to('cuda')

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

#### Breakdown

- `tokenizer.pad_token = tokenizer.eos_token` - this pads the end of sequence token

- `apply_chat_template()` - converts the conversational messages into models specific special tokens

- `return.tensors='pt'` - automatically tokenizes the formatted string and returns the numerical token IDs as a PyTorch Tensor

- `.to('cuda')` - moves the tensors onto the NVIDIA GPU

In [8]:
input

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    975,   5033,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    757,    264,  22380,    369,    279,
          14827,   2956,  57116, 128009]], device='cuda:0')

In [13]:
print("Decoding the Tokens back to words along with Llama's special tokens\n")
print(tokenizer.batch_decode(input))

Decoding the Tokens back to words along with Llama's special tokens

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 14 Aug 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell me a joke for the upcoming Data Scientists<|eot_id|>']


In [14]:
# quantizing the model

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map='auto', quantization_config=quant_config)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

#### Breakdown

- `AutoModelForCausalLM` - Selects the correct PyTorch model architecture. `CausalLM` means the model is trained to predict the next token in a sequence (autoregressive decoder only architecture)

- `from_pretrained(LLAMA)` - Downloads (if not already cached) and initializes the pre-trained weights and structure

- `device_map = 'auto'` - Uses huggingface library `accelerate` to handle memory allocation dynamically

  - It calculates vRAM needed for each layer and automatically maps models onto the available hardware

  - If you have a single GPU, it loads the model onto CUDA:0

  - If you have multiple GPUs, it splits layers across them

  - If a model is too large for GPU vRAM, it automatically offloads surplus layers onto the system CPU RAM or disk

- `quantization_config = quant_config` - to apply the quantization

In [15]:
memory = model.get_memory_footprint() / 1e6         # (total_size of all the model parameters in bytes / (10^6)) - to convert from bytes to MBs
print(f'Memory Footprint: {memory:,.1f} MB')        # , adds thousands separators and .1f rounds the float to 1 decimal value

Memory Footprint: 1,012.0 MB


## Looking under the hood at the Transformer model

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [16]:
# to investigate the layers

model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

#### **Notes about the Deep Neural Layers should be here**

### And if you want to go even deeper into Transformers

In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:  
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [18]:
# run the model

output = model.generate(input, max_new_tokens=100)
output[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,    975,   5033,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    757,    264,  22380,    369,    279,
         14827,   2956,  57116, 128009, 128006,  78191, 128007,    271,   8586,
           596,    264,  22380,    369,    279,   2956,  57116,   1473,  10445,
          1550,    279,    828,  28568,   1464,    709,    449,    813,  23601,
          1980,  18433,    568,   4934,    311,  24564,    872,   5133,    323,
          1505,    704,    422,    433,    574,   1120,    264,  26670,    477,
           264,   5353,   9976,  23937,   2268,   2028,  22380,  11335,    389,
           279,   7434,    315,    828,   6492,    323,  12135,     11,   3339,
           433,   9959,    311,    279,   2956,  57116,      6,   4913,     13,
        128009], device='cuda:0')

In [19]:
# decode the output token IDs back into words

print(tokenizer.decode(output[0]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 14 Aug 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell me a joke for the upcoming Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here's a joke for the Data Scientists:

Why did the data scientist break up with his girlfriend?

Because he wanted to analyze their relationship and find out if it was just a correlation or a cause-and-effect!

This joke plays on the concept of data analysis and relationships, making it relevant to the Data Scientists' profession.<|eot_id|>


In [ ]:
# clean up the memory (GPU vRAM) without having to restart the kernel

del model, tokenizer, input, output
gc.collect()
torch.cuda.empty_cache()

#### Breakdown

- `del` - to manually delete the variables

- `gc.collect()` - Even after deleting the variables, Python might not immediately destory the underlying C++ PyTorch tensor objects, especially if there are circular variable references or delayed cleanup

- `torch.cuda.empty_cache()` - forces PyTorch to release all unoccupied cached memory back to the GPU operating system so other models or processes can use it.

  - this is because, to speed up performance, PyTorch does not immediately return freed GPU memory back to the GPU hardware driver `nvidia-smi`. Instead, it keeps that VRAM cached for future tensor allocations.


#### Why All 3 are needed together?

- Without `del / gc.collect()`: PyTorch thinks the tensors are still active and will not release the memory, even if you call `torch.cuda.empty_cache()`.
- Without `torch.cuda.empty_cache()`: The memory is freed internally within PyTorch, but your GPU (nvidia-smi) will still report high memory usage because the allocator is holding onto the cache.

## A couple of quick notes on the next block of code:

I'm using a HuggingFace utility called TextStreamer so that results stream back.
To stream results, we simply replace:  
`output = model.generate(input, max_new_tokens=80)`  
With:  
`streamer = TextStreamer(tokenizer)`  
`output = model.generate(input, max_new_tokens=80, streamer=streamer)`

Also I've added the argument `add_generation_prompt=True` to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

Thank you to student Piotr B for raising the issue!

In [24]:
# Wrapping everything in a function - and adding streaming and generation prompts

messages = [
    {'role':'user', 'content':'Tell me a joke for the upcoming Data Scientists'}
    ]

quant_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4_bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type = 'nf4'
)

def generate(model, messages,quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device='cuda')
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to('cuda')
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to('cuda')
  output = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

In [25]:
generate(PHI, messages)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

<|user|>Tell me a joke for the upcoming Data Scientists<|end|><|assistant|>Sure, here's a joke tailored for data scientists:

Why did the data scientist break up with their computer?

Because they found out their partner was always "overfitting" to their every move!<|end|>


#### **Attention Mask**

`attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")`

This line creates a tensor of all 1s that matches the exact shape of your input_ids tensor, places it on the GPU, and assigns it to attention_mask.

It is used in Transformer models to tell the self-attention mechanism: "Pay attention to every single token in this input sequence—do not ignore or mask out anything."

1. `torch.ones_like(input_ids)`

- Inspects the shape (dimensions) of input_ids (e.g., [batch_size, sequence_length] or [1, 128]).
- Creates a brand new tensor with the exact same shape, filled entirely with the number 1.
2. `dtype=torch.long`
- Specifies the data type of the new tensor to be 64-bit integer (int64).
- Hugging Face models and PyTorch loss functions strictly expect integer types for indices and masks rather than floating-point numbers.
3. `device="cuda"`
- Allocates the new tensor directly on the NVIDIA GPU memory (cuda).
- Both input_ids and attention_mask must reside on the same device (e.g., both on cuda) for PyTorch matrix operations to run without throwing a runtime error.

**Why Is This Needed in Transformers?**

- In Transformer architectures (like Llama, BERT, GPT, or Mistral), an attention mask regulates where self-attention can look:
    - 1: Indicates a valid token that the model should process and attend to.
    - 0: Indicates a padded token (<pad>) that was added just to equalize batch lengths; the model completely ignores these during computation.

In [26]:
generate(QWEN, messages)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

<|im_start|>user
Tell me a joke for the upcoming Data Scientists<|im_end|>
<|im_start|>assistant
Sure! Here's a data scientist-approved joke with a dash of statistical humor:

---

Why did the data scientist break up with the statistician?

Because they kept *sampling* the relationship and never *pooled* the data properly — and honestly, it was just a *correlation* without causation!

---

Bonus: 📊 *P.S. We’re still analyzing the emotional impact…


In [27]:
# Reasoning Model. It keeps on putting wait in the middle of inference to continue the reasoning

generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<｜begin▁of▁sentence｜><｜User｜>Tell me a joke for the upcoming Data Scientists<｜Assistant｜><think>
Okay, so I need to come up with a joke for the upcoming Data Scientists. Hmm, let's think about what they do. They analyze data, find patterns, make predictions, optimize things, right? So, maybe something that involves numbers, statistics, or maybe even a bit of wordplay.

Wait, the user gave a joke already, so I should think of something else. Maybe something that's a bit more light-hearted or maybe something that's a bit more clever. Let's see. The previous joke was about calculating a number based on the number of letters in the word, which is a bit of a stretch. Maybe I can think of a different angle.

What if I think about something like a data science competition? They might use numbers or statistics in their puzzles. Or maybe something involving algorithms. Or perhaps something that's more about the process rather than the result.

Another angle could be something that involves the 

In [29]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}